### Probe DataCite award `sizes` parsing

Dry-run validator for the amount/currency parser added to `CreateDataCiteAwards`.

Reads `openalex.awards.datacite_award_items_deduplicated`, applies the parser inline (no writes, no DLT), and reports:

1. Coverage: how many records have `sizes[0]`, and of those how many parse successfully.
2. Currency distribution.
3. Sample of successful parses (raw → amount/currency).
4. Sample of **unparsed** sizes — non-null `sizes[0]` where the parser produced no result. These are the formats we may need to extend the parser for.

Run this before triggering the full DLT pipeline. If coverage looks reasonable and the unparsed bucket is dominated by truly non-monetary strings, the parser is good to ship.

In [ ]:
import pyspark.sql.functions as F

# Source: the deduplicated Award items table the DLT pipeline already produces.
# If you want to probe before that materialization exists, swap to reading
# `openalex.datacite.datacite_items` and re-apply the
# `attributes.types.resourceTypeGeneral == 'Award'` filter.
df = spark.read.table("openalex.awards.datacite_award_items_deduplicated")

# --- Identical parser to CreateDataCiteAwards ---
size_str = F.get(F.col("attributes.sizes"), 0)
iso_currency = F.upper(F.regexp_extract(size_str, r"(?i)\b([A-Z]{3})\b", 1))
symbol = F.regexp_extract(size_str, r"([\$€£¥])", 1)
symbol_currency = (
    F.when(symbol == "$", F.lit("USD"))
     .when(symbol == "€", F.lit("EUR"))
     .when(symbol == "£", F.lit("GBP"))
     .when(symbol == "¥", F.lit("JPY"))
)
parsed_currency = F.when(iso_currency != "", iso_currency).otherwise(symbol_currency)
amount_raw = F.regexp_extract(size_str, r"(\d[\d,]*(?:\.\d+)?)", 1)
parsed_amount = F.when(
    (amount_raw != "") & parsed_currency.isNotNull(),
    F.regexp_replace(amount_raw, ",", "").cast("double"),
)

probed = df.select(
    F.col("id").alias("doi"),
    size_str.alias("size_str"),
    parsed_amount.alias("amount"),
    parsed_currency.alias("currency"),
).cache()

probed.count()  # materialize the cache

#### 1. Coverage

In [ ]:
coverage = probed.agg(
    F.count("*").alias("total_award_records"),
    F.count("size_str").alias("with_sizes_field"),
    F.count("amount").alias("parsed_amount_currency"),
)
coverage.display()

# Same as a percent so you can eyeball it.
probed.agg(
    (F.count("size_str") * 100.0 / F.count("*")).alias("pct_with_sizes"),
    (F.count("amount") * 100.0 / F.count("size_str")).alias("pct_of_sizes_parsed"),
    (F.count("amount") * 100.0 / F.count("*")).alias("pct_of_all_records_parsed"),
).display()

#### 2. Currency distribution

In [ ]:
(
    probed.filter(F.col("currency").isNotNull())
    .groupBy("currency")
    .agg(
        F.count("*").alias("n"),
        F.round(F.avg("amount"), 0).alias("avg_amount"),
        F.min("amount").alias("min_amount"),
        F.max("amount").alias("max_amount"),
    )
    .orderBy(F.desc("n"))
    .display()
)

#### 3. Sample successful parses

Spot-check that the (raw size_str → amount, currency) mapping looks right. The known reference is `10.17920/G9C570` with `"263,614 USD"` → 263614.0, USD.

In [ ]:
(
    probed.filter(F.col("amount").isNotNull())
    .select("doi", "size_str", "amount", "currency")
    .limit(50)
    .display()
)

#### 4. Unparsed sizes

Rows where `sizes[0]` is non-null but the parser returned no amount/currency. Group by the raw string to see which formats are most common — that's the input for any follow-up extension to the parser.

If the top entries are clearly non-monetary (file sizes in bytes/MB, page counts, etc.), that's expected and we don't need to do anything. If we see a recurring monetary pattern we missed (e.g. `"$1.2M"`, `"approximately 100000 EUR"`), that's worth a parser tweak before the full run.

In [ ]:
(
    probed.filter(F.col("size_str").isNotNull() & F.col("amount").isNull())
    .groupBy("size_str")
    .agg(F.count("*").alias("n"), F.first("doi").alias("example_doi"))
    .orderBy(F.desc("n"))
    .limit(50)
    .display()
)

#### 5. Multi-element `sizes` arrays

We only read `sizes[0]`. If a meaningful share of records have multiple entries, the monetary value might live at a later index for some funders and we should iterate.

In [ ]:
(
    df.select(F.size(F.col("attributes.sizes")).alias("n_sizes"))
    .groupBy("n_sizes")
    .count()
    .orderBy("n_sizes")
    .display()
)